# 머리 탐지 학습 (Colab GPU)
추가 기능 Y①: COCO 사전학습 YOLO11n을 CrowdHuman **머리** 박스로 전이학습한다.
두 가지 구조를 학습해 비교한다.
- **head_base**: 원본 YOLO11n 구조, 클래스만 80 → 1 ('head')
- **head_p2**: 우리가 직접 수정한 구조 (`head_p2.yaml`, 작은 머리용 P2 탐지층 추가)

**시작 전 준비 (한 번만)**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`
2. 구글 드라이브에 `cv_project` 폴더를 만들고, 내 PC의 두 파일을 올려 둔다.
   - `src/convert_crowdhuman.py`
   - `models/head_p2.yaml`

학습 결과는 드라이브 `cv_project/runs/`에 저장되므로 세션이 끊겨도 이어서 학습할 수 있다.

## 1. GPU 확인 · 드라이브 연결 · 설치

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

In [ ]:
import os, shutil, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/cv_project')   # 내 파일·결과 저장 위치
WORK  = pathlib.Path('/content/cv_project')                 # Colab 로컬 디스크(빠름, 세션 끝나면 지워짐)
SAVE  = DRIVE / 'runs'                                      # 학습 결과 → 드라이브
for d in [WORK/'src', WORK/'models', WORK/'data/crowdhuman', SAVE]:
    d.mkdir(parents=True, exist_ok=True)
shutil.copy(DRIVE/'convert_crowdhuman.py', WORK/'src')
shutil.copy(DRIVE/'head_p2.yaml', WORK/'models')
os.chdir(WORK); print(os.listdir('src'), os.listdir('models'))

## 2. CrowdHuman 받기 (약 11GB, Hugging Face `sshao0516/CrowdHuman`)
비상업 연구·교육용(CC BY-NC 4.0), 재배포 금지 → Colab 로컬 디스크에만 두고 드라이브·GitHub에는 올리지 않는다.

In [ ]:
%cd /content/cv_project/data/crowdhuman
URL = 'https://huggingface.co/datasets/sshao0516/CrowdHuman/resolve/main/'
for f in ['annotation_train.odgt', 'annotation_val.odgt',
          'CrowdHuman_train01.zip', 'CrowdHuman_train02.zip', 'CrowdHuman_train03.zip', 'CrowdHuman_val.zip']:
    if not os.path.exists(f):
        !wget -q -c {URL}{f}
    if f.endswith('.zip'):
        !unzip -q -o {f} && rm {f}          # 모두 Images/ 폴더로 풀림. 디스크 아끼려고 zip은 지움
print(len(os.listdir('Images')), '장')        # 15000 + 4370 = 19370 이어야 함
%cd /content/cv_project

## 3. 라벨 변환 (우리가 작성한 `convert_crowdhuman.py`)
- 학습: 공식 train 15,000장 전체 / 검증: 공식 val 4,370장 전체
- mask·ignore 영역은 회색으로 칠해 지움 (이유는 스크립트 맨 위 설명 참고)

In [ ]:
!python src/convert_crowdhuman.py data/crowdhuman/annotation_train.odgt data/crowdhuman/Images train
!python src/convert_crowdhuman.py data/crowdhuman/annotation_val.odgt   data/crowdhuman/Images val
!cat data/head_yolo/head.yaml

## 4. 학습 설정
- `IMGSZ`: 머리가 작아서 기본값 640보다 크게. 첫 epoch 시간을 보고 조정한다.
- `BATCH`: 메모리 부족(CUDA out of memory)이 나면 8로 줄인다. 두 모델 비교를 위해 **같은 값**을 쓴다.

In [ ]:
from ultralytics import YOLO
DATA   = str(WORK/'data/head_yolo/head.yaml')
EPOCHS = 30
IMGSZ  = 960
BATCH  = 16
COMMON = dict(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=2,
              project=str(SAVE), exist_ok=True, seed=0)

## 5-A. head_base: 원본 구조, 클래스만 'head' 1개
`yolo11n.pt`(COCO)를 불러와 학습하면 Ultralytics가 Detect 층의 클래스 출력만 1개로 새로 만들고,
나머지 층은 COCO 가중치에서 시작한다(전이학습).

In [ ]:
YOLO('yolo11n.pt').train(name='head_base', **COMMON)

## 5-B. head_p2: 직접 수정한 구조 + COCO 가중치 이식
`head_p2.yaml`로 구조를 만든 뒤 `yolo11n.pt`에서 모양이 맞는 층(백본 등)만 옮겨 받는다.
새로 추가한 P2 경로와 Detect 층은 처음부터 학습된다.

In [ ]:
YOLO('models/head_p2.yaml').load('yolo11n.pt').train(name='head_p2', **COMMON)

## 세션이 끊겼을 때: 이어서 학습
1번 셀 두 개를 다시 실행한 다음, 끊긴 모델의 줄만 실행한다. (데이터를 다시 받을 필요는 없지만, 새 세션이면 2·3번도 다시 실행해야 함)

In [ ]:
# YOLO(str(SAVE/'head_base/weights/last.pt')).train(resume=True)
# YOLO(str(SAVE/'head_p2/weights/last.pt')).train(resume=True)

## 6. 결과 비교 (검증 4,370장)

In [ ]:
import pandas as pd
rows = []
for name in ['head_base', 'head_p2']:
    w = SAVE/name/'weights/best.pt'
    if not w.exists(): continue
    m = YOLO(str(w)); r = m.val(data=DATA, imgsz=IMGSZ, batch=BATCH, verbose=False)
    rows.append(dict(model=name, params=sum(p.numel() for p in m.model.parameters()),
                     P=r.box.mp, R=r.box.mr, mAP50=r.box.map50, mAP50_95=r.box.map,
                     ms_per_img=r.speed['inference']))
pd.DataFrame(rows).round(4)

## 7. 가중치 가져오기
드라이브 `cv_project/runs/<이름>/weights/best.pt`를 내 PC로 내려받아 `models/`에 둔다.
- `head_base/weights/best.pt` → `models/head_base.pt`
- `head_p2/weights/best.pt`   → `models/head_p2.pt`